In [14]:
# ==========================================================
# ECOTHON PIPELINE
# Satellite Image + Sentinel5P Dataset
# Kiln Detection → Emission Score → AQI Forecast → Map
# ==========================================================

!pip install ultralytics folium torch torchvision scikit-learn pandas numpy opencv-python

import os
import cv2
import torch
import numpy as np
import pandas as pd
import folium

from google.colab import drive, files
from ultralytics import YOLO
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn

# ==========================================================
# 1. Mount Google Drive
# ==========================================================

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Ecothon_AQI"

folders = ["dataset","models","results","maps"]

for f in folders:
    os.makedirs(f"{BASE}/{f}",exist_ok=True)

print("Folders ready")

# ==========================================================
# 2. Load Sentinel-5P Dataset
# ==========================================================

DATA_PATH = "/content/drive/MyDrive/Sentinel5P_Pollution_2025_2026.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded")
print("Total rows:",len(df))

# Select pollution features
pollution_features = ["NO2","SO2","CO"]

data = df[pollution_features].fillna(0)

# Calculate AQI from available pollution features
# Using arbitrary weights to create a composite AQI score.
# These weights would ideally be based on EPA/WHO standards or similar.
# Assuming a simple linear combination for demonstration.
df["AQI"] = (
    df["NO2"] * 100000 +  # Example scaling for NO2
    df["SO2"] * 100000 +  # Example scaling for SO2
    df["CO"] * 100        # Example scaling for CO
)

# Ensure AQI values are non-negative
df["AQI"] = df["AQI"].clip(lower=0)

# ==========================================================
# 3. Upload Satellite Image
# ==========================================================

print("Upload satellite image")

uploaded = files.upload()

for name in uploaded.keys():

    img_path = f"{BASE}/dataset/{name}"

    with open(img_path,"wb") as f:
        f.write(uploaded[name])

print("Image saved:",img_path)

# ==========================================================
# 4. Load Image
# ==========================================================

image = cv2.imread(img_path)
image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
image = cv2.resize(image,(640,640))

# ==========================================================
# 5. Industrial / Kiln Detection
# ==========================================================

model = YOLO("yolov8n.pt")

results = model.predict(image)

detections=[]

for r in results:

    boxes = r.boxes.xyxy.cpu().numpy()

    for box in boxes:

        x1,y1,x2,y2 = box

        area = (x2-x1)*(y2-y1)

        detections.append([x1,y1,x2,y2,area])

print("Detected industries:",len(detections))

# ==========================================================
# 6. Emission Score using Sentinel dataset
# ==========================================================

mean_pollution = data.mean()

emission_base = (
    mean_pollution["NO2"] +
    mean_pollution["SO2"] +
    mean_pollution["CO"]
)/3

emissions=[]
coordinates=[]

for d in detections:

    x1,y1,x2,y2,area=d

    score = emission_base*(area/(640*640))

    emissions.append(score)

    cx=(x1+x2)/2
    cy=(y1+y2)/2

    lat = 20 + (cy/640)*10
    lon = 70 + (cx/640)*10

    coordinates.append([lat,lon])

# ==========================================================
# 7. AQI Forecast Model (LSTM)
# ==========================================================

class AQILSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm=nn.LSTM(1,32,2,batch_first=True)

        self.fc=nn.Linear(32,1)

    def forward(self,x):

        x,_=self.lstm(x)

        x=self.fc(x[:,-1,:])

        return x

lstm=AQILSTM()

aqi_series=df["AQI"].fillna(method="ffill").values

scaler=MinMaxScaler()

aqi_scaled=scaler.fit_transform(aqi_series.reshape(-1,1))

X=[]
y=[]

for i in range(len(aqi_scaled)-1):

    X.append(aqi_scaled[i])
    y.append(aqi_scaled[i+1])

X=torch.tensor(X).float().unsqueeze(-1)
y=torch.tensor(y).float()

optimizer=torch.optim.Adam(lstm.parameters(),lr=0.01)
loss_fn=nn.MSELoss()

for epoch in range(20):

    pred=lstm(X)

    loss=loss_fn(pred,y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

pred=lstm(X[-1:].detach())

aqi_pred=scaler.inverse_transform(pred.detach().numpy())

print("Predicted AQI:",aqi_pred[0][0])

# ==========================================================
# 8. Save Results
# ==========================================================

df_out=pd.DataFrame({
    "latitude":[c[0] for c in coordinates],
    "longitude":[c[1] for c in coordinates],
    "emission_score":emissions
})

csv_path=f"{BASE}/results/kiln_emissions.csv"

df_out.to_csv(csv_path,index=False)

print("Results saved:",csv_path)

# ==========================================================
# 9. Detection Visualization
# ==========================================================

vis=image.copy()

for d in detections:

    x1,y1,x2,y2,_=d

    cv2.rectangle(vis,(int(x1),int(y1)),(int(x2),int(y2)),(255,0,0),2)

cv2.imwrite(f"{BASE}/results/detection.png",cv2.cvtColor(vis,cv2.COLOR_RGB2BGR))

# ==========================================================
# 10. Interactive Pollution Map
# ==========================================================

if len(coordinates)>0:

    m=folium.Map(location=coordinates[0],zoom_start=6)

    for i,c in enumerate(coordinates):

        folium.CircleMarker(
            location=c,
            radius=6,
            color="red",
            popup=f"Emission Score:{emissions[i]}"
        ).add_to(m)

    map_path=f"{BASE}/maps/pollution_map.html"

    m.save(map_path)

    print("Map saved:",map_path)

# ==========================================================
# 11. Save Model
# ==========================================================

torch.save(lstm.state_dict(),f"{BASE}/models/aqi_lstm_model.pth")

print("Pipeline completed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready
Dataset loaded
Total rows: 365
Upload satellite image


Saving Satellite-image-of-the-cluster-of-brick-kilns-of-the-sampling-site-10-Total-Suspended.webp to Satellite-image-of-the-cluster-of-brick-kilns-of-the-sampling-site-10-Total-Suspended (1).webp
Image saved: /content/drive/MyDrive/Ecothon_AQI/dataset/Satellite-image-of-the-cluster-of-brick-kilns-of-the-sampling-site-10-Total-Suspended (1).webp

0: 640x640 2 boats, 10.1ms
Speed: 1.7ms preprocess, 10.1ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
Detected industries: 2


/tmp/ipykernel_10813/3761473503.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  aqi_series=df["AQI"].fillna(method="ffill").values
/tmp/ipykernel_10813/3761473503.py:183: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X=torch.tensor(X).float().unsqueeze(-1)


Predicted AQI: 12.838908
Results saved: /content/drive/MyDrive/Ecothon_AQI/results/kiln_emissions.csv
Map saved: /content/drive/MyDrive/Ecothon_AQI/maps/pollution_map.html
Pipeline completed
